## Undersampling to prepare data for XAI validation

In [1]:
import numpy as np
import pandas as pd

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

In [2]:
first_level_dir = 'G:/Informatica/IA/enzimas/datasets/First Level/test_features_sequences.csv'
second_level_dir = 'G:/Informatica/IA/enzimas/datasets/Second_Level/test_features_sequences.csv'
third_level_dir = 'G:/Informatica/IA/enzimas/datasets/test_features_sequences.csv'
y_first_dir = 'G:/Informatica/IA/enzimas/datasets/First Level/y_test.npy'
y_second_dir = 'G:/Informatica/IA/enzimas/datasets/Second_Level/y_test.npy'
y_third_dir = 'G:/Informatica/IA/enzimas/datasets/pre_processed/y_test.npy'


In [3]:
data_first = pd.read_csv(first_level_dir)
data_second = pd.read_csv(second_level_dir)
data_third = pd.read_csv(third_level_dir)
first_emb = data_first.iloc[:, :1024].to_numpy(dtype=np.float32)
first_feat = data_first.iloc[:, 1024:-1].to_numpy(dtype=np.float32)
second_emb = data_second.iloc[:, :1024].to_numpy(dtype=np.float32)
second_feat = data_second.iloc[:, 1024:-1].to_numpy(dtype=np.float32)
third_emb = data_third.iloc[:, :1024].to_numpy(dtype=np.float32)
third_feat = data_third.iloc[:, 1024:-1].to_numpy(dtype=np.float32)

In [4]:
y_first = np.load(y_first_dir)
y_second = np.load(y_second_dir)
y_third = np.load(y_third_dir)

In [5]:
def _flatten_labels(labels):
    labels = np.asarray(labels)

    if labels.ndim == 1:
        return labels

    if labels.ndim == 2:
        if labels.shape[1] == 1:
            return labels.ravel()
        return labels.argmax(axis=1)

    raise ValueError('labels debe ser un vector 1D o una matriz one-hot 2D.')


def _lookup_class_weight(class_weights, cls):
    if cls in class_weights:
        return class_weights[cls]

    try:
        cls_int = int(cls)
        if cls_int in class_weights:
            return class_weights[cls_int]
        cls_str = str(cls_int)
        if cls_str in class_weights:
            return class_weights[cls_str]
    except (TypeError, ValueError):
        pass

    cls_str = str(cls)
    if cls_str in class_weights:
        return class_weights[cls_str]

    raise KeyError(f'No se encontr\u00f3 un peso para la clase {cls!r}.')


def recommend_class_weights(labels, n_select, strategy='proportional'):
    """
    Genera cuotas sugeridas por clase para usar en class_weights.

    Estrategias disponibles
    ----------------------
    - 'proportional': conserva la proporci\u00f3n original (devuelve None).
    - 'balanced': reparte el presupuesto de forma aproximadamente uniforme.
    - 'keep_all_minority': solo para binario; conserva toda la clase minoritaria
      y llena el resto con la mayoritaria.
    """
    labels = _flatten_labels(labels)
    classes, counts = np.unique(labels, return_counts=True)
    k = int(n_select)

    if k < len(classes):
        raise ValueError('n_select debe ser al menos igual al n\u00famero de clases.')

    if strategy == 'proportional':
        return None

    if strategy == 'balanced':
        quotas = np.full(len(classes), k // len(classes), dtype=int)
        quotas[:k % len(classes)] += 1
        quotas = np.minimum(quotas, counts)

        missing = k - int(quotas.sum())
        while missing > 0:
            capacity = counts - quotas
            available = np.flatnonzero(capacity > 0)
            if len(available) == 0:
                break
            best = available[int(np.argmax(capacity[available]))]
            quotas[best] += 1
            missing -= 1

        return dict(zip(classes.tolist(), quotas.tolist()))

    if strategy == 'keep_all_minority':
        if len(classes) != 2:
            raise ValueError("'keep_all_minority' solo est\u00e1 disponible para clasificaci\u00f3n binaria.")

        minority_pos = int(np.argmin(counts))
        majority_pos = 1 - minority_pos

        minority_count = int(counts[minority_pos])
        majority_count = int(counts[majority_pos])

        if minority_count > k:
            raise ValueError(
                'La clase minoritaria tiene m\u00e1s muestras que el presupuesto total; no se puede conservar completa.'
            )

        quotas = np.zeros(2, dtype=int)
        quotas[minority_pos] = minority_count
        quotas[majority_pos] = k - minority_count

        if quotas[majority_pos] > majority_count:
            raise ValueError('No hay suficientes muestras en la clase mayoritaria para completar el presupuesto.')

        return dict(zip(classes.tolist(), quotas.tolist()))

    raise ValueError(f'Estrategia no soportada: {strategy!r}')


def _normalize_class_weights(labels, class_weights=None):
    classes, counts = np.unique(labels, return_counts=True)

    if class_weights is None:
        raw = counts.astype(float)
    elif isinstance(class_weights, dict):
        raw = np.array([_lookup_class_weight(class_weights, cls) for cls in classes], dtype=float)
    else:
        raw = np.asarray(class_weights, dtype=float)
        if raw.shape[0] != len(classes):
            raise ValueError(
                f'class_weights debe tener longitud {len(classes)} (una por clase en np.unique(labels)).'
            )

    if np.any(raw < 0):
        raise ValueError('class_weights no puede contener valores negativos.')
    if raw.sum() == 0:
        raise ValueError('class_weights no puede sumar 0.')

    target = raw / raw.sum()
    return classes, counts.astype(int), target


def _allocate_quotas(labels, k, class_weights=None):
    labels = _flatten_labels(labels)
    classes, counts, target = _normalize_class_weights(labels, class_weights=class_weights)

    if k > len(labels):
        raise ValueError('k no puede ser mayor que el n\u00famero total de muestras.')
    if k < len(classes):
        raise ValueError(
            f'k={k} es menor que el n\u00famero de clases={len(classes)}. No es posible asignar al menos una muestra por clase.'
        )

    raw = target * k
    quotas = np.floor(raw).astype(int)
    quotas = np.maximum(quotas, 1)
    quotas = np.minimum(quotas, counts)

    diff = k - int(quotas.sum())

    while diff > 0:
        capacity = counts - quotas
        candidates = np.flatnonzero(capacity > 0)
        if len(candidates) == 0:
            raise RuntimeError('No quedan clases con capacidad para completar el cupo total.')

        priority = raw[candidates] - quotas[candidates]
        best = candidates[int(np.argmax(priority))]
        quotas[best] += 1
        diff -= 1

    while diff < 0:
        removable = np.flatnonzero(quotas > 1)
        if len(removable) == 0:
            raise RuntimeError('No es posible reducir cuotas sin dejar una clase vac\u00eda.')

        surplus = quotas[removable] - raw[removable]
        best = removable[int(np.argmax(surplus))]
        quotas[best] -= 1
        diff += 1

    if int(quotas.sum()) != k:
        raise RuntimeError(f'No se pudo repartir exactamente k={k}. Se obtuvo {quotas.sum()}.')

    return classes, quotas.astype(int), counts.astype(int)


def _prepare_joint_space(
    descriptors,
    embeddings,
    pca_desc_components=50,
    pca_emb_components=50,
    descriptor_weight=1.0,
    embedding_weight=1.0,
    random_state=42,
):
    descriptors = np.asarray(descriptors, dtype=np.float32)
    embeddings = np.asarray(embeddings, dtype=np.float32)

    if descriptors.shape[0] != embeddings.shape[0]:
        raise ValueError('descriptors y embeddings deben tener el mismo n\u00famero de filas.')

    desc_scaled = StandardScaler().fit_transform(descriptors)
    emb_scaled = StandardScaler().fit_transform(embeddings)

    if pca_desc_components is not None and desc_scaled.shape[1] > pca_desc_components:
        n_comp = min(pca_desc_components, desc_scaled.shape[0] - 1, desc_scaled.shape[1])
        if n_comp >= 1:
            desc_scaled = PCA(n_components=n_comp, random_state=random_state).fit_transform(desc_scaled)

    if pca_emb_components is not None and emb_scaled.shape[1] > pca_emb_components:
        n_comp = min(pca_emb_components, emb_scaled.shape[0] - 1, emb_scaled.shape[1])
        if n_comp >= 1:
            emb_scaled = PCA(n_components=n_comp, random_state=random_state).fit_transform(emb_scaled)

    desc_scaled = desc_scaled / max(np.sqrt(desc_scaled.shape[1]), 1.0)
    emb_scaled = emb_scaled / max(np.sqrt(emb_scaled.shape[1]), 1.0)

    joint_space = np.concatenate(
        [descriptor_weight * desc_scaled, embedding_weight * emb_scaled],
        axis=1,
    ).astype(np.float32)

    return joint_space


def _squared_euclidean_distances(X, Y):
    X = np.asarray(X, dtype=np.float32)
    Y = np.asarray(Y, dtype=np.float32)

    X_norm = np.sum(X * X, axis=1, keepdims=True)
    Y_norm = np.sum(Y * Y, axis=1, keepdims=True).T
    distances = X_norm + Y_norm - 2.0 * (X @ Y.T)
    return np.maximum(distances, 0.0)


def _assign_to_medoids(X, medoid_points, chunk_size=2048):
    n_samples = X.shape[0]
    assignments = np.empty(n_samples, dtype=int)
    min_distances = np.empty(n_samples, dtype=np.float32)
    total_cost = 0.0

    for start in range(0, n_samples, chunk_size):
        stop = min(start + chunk_size, n_samples)
        dist_sq = _squared_euclidean_distances(X[start:stop], medoid_points)
        local_assign = np.argmin(dist_sq, axis=1)
        local_min_sq = dist_sq[np.arange(stop - start), local_assign]
        local_min = np.sqrt(local_min_sq, dtype=np.float32)

        assignments[start:stop] = local_assign
        min_distances[start:stop] = local_min
        total_cost += float(local_min.sum())

    return assignments, min_distances, total_cost


def _kpp_medoid_init(X, k, random_state=42):
    rng = np.random.default_rng(random_state)
    n_samples = X.shape[0]

    if k >= n_samples:
        return np.arange(n_samples, dtype=int)

    centroid = X.mean(axis=0, keepdims=True)
    first = int(np.argmin(_squared_euclidean_distances(X, centroid).ravel()))

    selected = [first]
    min_dist_sq = _squared_euclidean_distances(X, X[[first]]).ravel()
    min_dist_sq[first] = 0.0

    while len(selected) < k:
        probs = min_dist_sq.copy()
        probs[selected] = 0.0

        total = float(probs.sum())
        if total <= 0:
            remaining = np.setdiff1d(np.arange(n_samples), np.array(selected), assume_unique=False)
            next_idx = int(rng.choice(remaining))
        else:
            next_idx = int(rng.choice(n_samples, p=probs / total))

        selected.append(next_idx)
        next_dist_sq = _squared_euclidean_distances(X, X[[next_idx]]).ravel()
        min_dist_sq = np.minimum(min_dist_sq, next_dist_sq)
        min_dist_sq[selected] = 0.0

    return np.array(selected, dtype=int)


def _alternate_k_medoids(X, k, max_iter=25, random_state=42):
    n_samples = X.shape[0]

    if k >= n_samples:
        return np.arange(n_samples, dtype=int), 0.0

    medoid_idx = _kpp_medoid_init(X, k, random_state=random_state)

    for _ in range(max_iter):
        assignments, min_distances, _ = _assign_to_medoids(X, X[medoid_idx])
        proposal = []

        for cluster_id in range(k):
            members = np.flatnonzero(assignments == cluster_id)
            if len(members) == 0:
                continue
            if len(members) == 1:
                proposal.append(int(members[0]))
                continue

            intra_sq = _squared_euclidean_distances(X[members], X[members])
            intra_cost = np.sqrt(intra_sq, dtype=np.float32).sum(axis=1)
            proposal.append(int(members[int(np.argmin(intra_cost))]))

        proposal = list(dict.fromkeys(proposal))

        if len(proposal) < k:
            candidate_order = np.argsort(min_distances)[::-1]
            taken = set(proposal)
            for idx in candidate_order:
                idx = int(idx)
                if idx in taken:
                    continue
                proposal.append(idx)
                taken.add(idx)
                if len(proposal) == k:
                    break

        new_medoids = np.array(proposal[:k], dtype=int)

        if np.array_equal(np.sort(new_medoids), np.sort(medoid_idx)):
            medoid_idx = new_medoids
            break

        medoid_idx = new_medoids

    _, _, final_cost = _assign_to_medoids(X, X[medoid_idx])
    return medoid_idx, final_cost


def _resolve_clara_sample_size(n_samples, k, clara_sample_size):
    if clara_sample_size == 'auto':
        clara_sample_size = max(40 + 2 * k, k + 2)
    clara_sample_size = int(clara_sample_size)
    clara_sample_size = max(clara_sample_size, k + 2)
    clara_sample_size = min(clara_sample_size, n_samples)
    return clara_sample_size


def _clara_k_medoids(
    X,
    k,
    clara_sample_size='auto',
    clara_trials=5,
    kmedoids_max_iter=25,
    random_state=42,
):
    n_samples = X.shape[0]

    if k >= n_samples:
        return np.arange(n_samples, dtype=int), 0.0

    sample_size = _resolve_clara_sample_size(n_samples, k, clara_sample_size)
    rng = np.random.default_rng(random_state)

    best_medoids = None
    best_cost = np.inf

    for trial in range(clara_trials):
        sample_idx = rng.choice(n_samples, size=sample_size, replace=False)
        local_medoids, _ = _alternate_k_medoids(
            X[sample_idx],
            k=k,
            max_iter=kmedoids_max_iter,
            random_state=random_state + trial,
        )
        candidate_medoids = sample_idx[local_medoids]
        _, _, candidate_cost = _assign_to_medoids(X, X[candidate_medoids])

        if candidate_cost < best_cost:
            best_cost = candidate_cost
            best_medoids = candidate_medoids.copy()

    best_medoids = np.array(sorted(set(best_medoids.tolist())), dtype=int)

    if best_medoids.shape[0] != k:
        raise RuntimeError('CLARA no devolvi\u00f3 exactamente k medoids \u00fanicos.')

    return best_medoids, float(best_cost)


def stratified_clara_kmedoids_subsample(
    descriptors,
    embeddings,
    labels,
    n_select,
    class_weights=None,
    pca_desc_components=50,
    pca_emb_components=50,
    descriptor_weight=1.0,
    embedding_weight=1.0,
    clara_sample_size='auto',
    clara_trials=5,
    kmedoids_max_iter=25,
    random_state=42,
    return_metadata=False,
):
    labels = _flatten_labels(labels)
    n_total = len(labels)

    if isinstance(n_select, float):
        if not (0 < n_select <= 1):
            raise ValueError('Si n_select es float, debe estar en (0, 1].')
        k = int(np.round(n_total * n_select))
    else:
        k = int(n_select)

    k = max(1, min(k, n_total))

    X = _prepare_joint_space(
        descriptors,
        embeddings,
        pca_desc_components=pca_desc_components,
        pca_emb_components=pca_emb_components,
        descriptor_weight=descriptor_weight,
        embedding_weight=embedding_weight,
        random_state=random_state,
    )

    classes, quotas, counts = _allocate_quotas(labels, k, class_weights=class_weights)

    selected_indices = []
    class_rows = []

    for offset, (cls, quota, count) in enumerate(zip(classes, quotas, counts)):
        cls_idx = np.flatnonzero(labels == cls)

        if quota >= len(cls_idx):
            chosen = cls_idx.copy()
            class_cost = 0.0
        else:
            local_medoids, class_cost = _clara_k_medoids(
                X[cls_idx],
                k=quota,
                clara_sample_size=clara_sample_size,
                clara_trials=clara_trials,
                kmedoids_max_iter=kmedoids_max_iter,
                random_state=random_state + offset,
            )
            chosen = cls_idx[local_medoids]

        selected_indices.extend(chosen.tolist())
        class_rows.append(
            {
                'class': cls,
                'available': int(count),
                'selected': int(len(chosen)),
                'selection_rate': float(len(chosen) / count),
                'clara_cost': float(class_cost),
            }
        )

    selected_indices = np.array(sorted(selected_indices), dtype=int)

    if selected_indices.shape[0] != k:
        raise RuntimeError(
            f'La selecci\u00f3n final deber\u00eda contener {k} muestras y devolvi\u00f3 {selected_indices.shape[0]}.'
        )

    class_summary = pd.DataFrame(class_rows)

    if return_metadata:
        return {
            'selected_indices': selected_indices,
            'class_summary': class_summary,
            'joint_space_shape': X.shape,
            'quotas': dict(zip(classes.tolist(), quotas.tolist())),
        }

    return selected_indices

### 1st level

In [16]:
first_level_weights = recommend_class_weights(
    y_first,
    n_select=300,
)

selection = stratified_clara_kmedoids_subsample(
    descriptors=first_feat,
    embeddings=first_emb,
    labels=y_first,
    n_select=300,
    class_weights=first_level_weights,
    pca_desc_components=50,
    pca_emb_components=50,
    descriptor_weight=1.0,
    embedding_weight=1.0,
    clara_sample_size='auto',
    clara_trials=5,
    kmedoids_max_iter=25,
    random_state=42,
    return_metadata=True,
)

indexes = selection['selected_indices']
selected_labels = _flatten_labels(y_first)[indexes]
y_first = y_first[indexes]
selected_subset = data_first.iloc[indexes].copy()
selected_subset.to_csv('G:/Informatica/IA/enzimas/datasets/First Level/1l_test_xai.csv')
np.save('G:/Informatica/IA/enzimas/datasets/First Level/1l_y_xai.csv', y_first)
selected_subset.insert(0, 'original_index', indexes)

In [19]:
print(f'Total de muestras seleccionadas: {len(indexes)}')
print('Dimensión del espacio conjunto:', selection['joint_space_shape'])
print('Cuotas utilizadas:', selection['quotas'])
print('\nResumen por clase:')
display(selection['class_summary'])

print('\nPrimeros índices seleccionados:')
print(indexes[:20])

print('\nPrimeras secuencias seleccionadas:')
display(selected_subset[['original_index', 'sequence']].head())

Total de muestras seleccionadas: 300
Dimensión del espacio conjunto: (22046, 100)
Cuotas utilizadas: {0: 174, 1: 126}

Resumen por clase:


,class,available,selected,selection_rate,clara_cost
0,0,12755,174,0.013642,51896.932861
1,1,9291,126,0.013562,27755.749023



Primeros índices seleccionados:
[  26   65   67   88  119  231  278  366  504  665  851  926  937 1094
 1103 1223 1226 1268 1464 1587]

Primeras secuencias seleccionadas:


,original_index,sequence
26,26,MRSPRVLGLVLAGGAGKRLAPLTADRAKPAVPFGGIYRLVDFVLSN...
65,65,MAEITASLVKELRERTGAGMMDCKKALTEANGDIELAIENMRKSGA...
67,67,MSPNSEATGTAWAPPPPRPSRGVIMISSVSTNDVRRFLLCMRVYST...
88,88,MANHKSAEKRIRQIKKRTEINRARISRIRTFVKKVELAIASGDKAA...
119,119,MHLNQLVISHLACERGENRLFEGCHFSVSSGEWVQIEGHNGIGKTS...


### 2nd level

In [20]:
second_level_weights = recommend_class_weights(
    y_second,
    n_select=300,
    strategy='balanced',
)

selection = stratified_clara_kmedoids_subsample(
    descriptors=second_feat,
    embeddings=second_emb,
    labels=y_second,
    n_select=300,
    class_weights=second_level_weights,
    pca_desc_components=50,
    pca_emb_components=50,
    descriptor_weight=1.0,
    embedding_weight=1.0,
    clara_sample_size='auto',
    clara_trials=5,
    kmedoids_max_iter=25,
    random_state=42,
    return_metadata=True,
)

indexes = selection['selected_indices']
selected_labels = _flatten_labels(y_second)[indexes]
selected_subset = data_second.iloc[indexes].copy()
selected_subset.to_csv('G:/Informatica/IA/enzimas/datasets/Second_Level/2l_test_xai.csv')
np.save('G:/Informatica/IA/enzimas/datasets/Second_Level/2l_y_xai.csv', y_second)
selected_subset.insert(0, 'original_index', indexes)

In [22]:
print(f'Total de muestras seleccionadas: {len(indexes)}')
print('Dimensión del espacio conjunto:', selection['joint_space_shape'])
print('Cuotas utilizadas:', selection['quotas'])
print('\nResumen por clase:')
display(selection['class_summary'])

print('\nPrimeros índices seleccionados:')
print(indexes[:20])

print('\nPrimeras secuencias seleccionadas:')
display(selected_subset[['original_index', 'sequence']].head())

Total de muestras seleccionadas: 300
Dimensión del espacio conjunto: (18582, 100)
Cuotas utilizadas: {0: 43, 1: 43, 2: 43, 3: 43, 4: 43, 5: 43, 6: 42}

Resumen por clase:


,class,available,selected,selection_rate,clara_cost
0,0,2261,43,0.019018,8846.865906
1,1,6759,43,0.006362,27686.184570
2,2,3453,43,0.012453,15607.911133
3,3,1913,43,0.022478,7153.391602
4,4,1196,43,0.035953,4213.598633
5,5,2263,43,0.019001,7524.915588
6,6,737,42,0.056988,2395.363281



Primeros índices seleccionados:
[  75   82   97  138  302  325  357  381  389  430  733  743  826  879
  943  952  992  996 1036 1095]

Primeras secuencias seleccionadas:


,original_index,sequence
75,75,MEKFLDINEIKKIIPHRYPFLLVDKITELEEGKSAVGYKNVTANEY...
82,82,MTVPLYADAREAIACKTRFVPDFPVPGVIFEDLTPVLADAAAFSLI...
97,97,MLLTMMKAKLHRATVTQADLDYEGSIAIDTDLLDAAGILPNEQVDV...
138,138,MQNMVILGATGSIGASTLSVISANPLAYSVYGLVANASVDKMLALC...
302,302,MNALPPAIFLMGPTAAGKTDLAIELSKVLPCELISVDSALVYRGMD...


### 3rd level

In [6]:
# Ejemplo de uso para el 3er nivel (binario con fuerte desbalance).
# Esta estrategia conserva toda la clase minoritaria y selecciona medoids
# representativos de la clase mayoritaria hasta completar las 300 muestras.
#
# Ejemplos para otros escenarios:
# - Binario aproximadamente proporcional: class_weights = recommend_class_weights(y, 300, strategy='proportional')
# - Binario balanceado: class_weights = recommend_class_weights(y, 300, strategy='balanced')
# - Multiclase balanceado: class_weights = recommend_class_weights(y_multiclass, 300, strategy='balanced')
# - Multiclase con cuotas manuales: class_weights = {0: 40, 1: 40, 2: 45, 3: 45, 4: 45, 5: 45, 6: 40}

third_level_weights = recommend_class_weights(
    y_third,
    n_select=350,
    strategy='keep_all_minority',
)

selection = stratified_clara_kmedoids_subsample(
    descriptors=third_feat,
    embeddings=third_emb,
    labels=y_third,
    n_select=350,
    class_weights=third_level_weights,
    pca_desc_components=50,
    pca_emb_components=50,
    descriptor_weight=1.0,
    embedding_weight=1.0,
    clara_sample_size='auto',
    clara_trials=5,
    kmedoids_max_iter=25,
    random_state=42,
    return_metadata=True,
)

indexes = selection['selected_indices']
selected_labels = _flatten_labels(y_third)[indexes]
selected_subset = data_third.iloc[indexes].copy()
selected_subset.to_csv('G:/Informatica/IA/enzimas/datasets/3l_test_xai.csv')
np.save('G:/Informatica/IA/enzimas/datasets/3l_y_xai.csv', y_third)
selected_subset.insert(0, 'original_index', indexes)

In [7]:
print(f'Total de muestras seleccionadas: {len(indexes)}')
print('Dimensión del espacio conjunto:', selection['joint_space_shape'])
print('Cuotas utilizadas:', selection['quotas'])
print('\nResumen por clase:')
display(selection['class_summary'])

print('\nPrimeros índices seleccionados:')
print(indexes[:20])

print('\nPrimeras secuencias seleccionadas:')
display(selected_subset[['original_index', 'sequence']].head())

# Si quieres persistir el subconjunto para XAI, puedes descomentar esta l\u00ednea.
# selected_subset.to_csv('datasets/xai_selected_subset.csv', index=False)

Total de muestras seleccionadas: 350
Dimensión del espacio conjunto: (3079, 100)
Cuotas utilizadas: {0.0: 304, 1.0: 46}

Resumen por clase:


,class,available,selected,selection_rate,clara_cost
0,0.0,3033,304,0.100231,9446.122559
1,1.0,46,46,1.000000,0.000000



Primeros índices seleccionados:
[  3   4   9  29  34  35  44  45  52  55  64  77  84  91  99 108 112 126
 129 130]

Primeras secuencias seleccionadas:


,original_index,sequence
3,3,MSWDQVWIDVNVATMDPSISAPYGAITHAAIAVKDGKIAWLGPRSE...
4,4,MAKEPTKAAHPQPEQTKTNHKAHRPVGGYVLAKDPIEINQGRPRTT...
9,9,MVLLYSQASWDKRSKADALVLPFWMKNSKAQEAAVVDEDYKLVYQN...
29,29,MPNIEELKQRQEKIRNFSIIAHIDHGKSTLADRILEKTETVSSREM...
34,34,MLSLILFFPSFAFAATPVTPYYGPGHITFDWCGFGDSRSDCTNPQS...


## Getting Background for SHAP (per level)

### 1st level

In [2]:
ptf_path = 'G:/Informatica/IA/enzimas/datasets/First Level/pre_processed/train_feat.npy'
pte_path = 'G:/Informatica/IA/enzimas/datasets/First Level/pre_processed/train_emb.npy'
y_train_path = 'G:/Informatica/IA/enzimas/datasets/First Level/y_train.npy'
df_train_path = 'G:/Informatica/IA/enzimas/datasets/First Level/train_descriptores.csv'

In [4]:
first_emb = np.load(pte_path)
first_feat = np.load(ptf_path)
y_first = np.load(y_train_path)

In [ ]:
first_level_weights = recommend_class_weights(
    y_first,
    n_select=50,
)

selection = stratified_clara_kmedoids_subsample(
    descriptors=first_feat,
    embeddings=first_emb,
    labels=y_first,
    n_select=50,
    class_weights=first_level_weights,
    pca_desc_components=50,
    pca_emb_components=50,
    descriptor_weight=1.0,
    embedding_weight=1.0,
    clara_sample_size='auto',
    clara_trials=5,
    kmedoids_max_iter=25,
    random_state=42,
    return_metadata=True,
)

indexes = selection['selected_indices']
np.save('G:/Informatica/IA/enzimas/datasets/First Level/index_for_background.npy', indexes)
y_first = y_first[indexes]
np.save('G:/Informatica/IA/enzimas/datasets/First Level/1l_y_background.npy', y_first)
first_feat_f = first_feat[indexes]
first_emb_t = first_emb[indexes]
selected = pd.DataFrame(np.concatenate((first_emb_t, first_feat_f), axis=1))
selected.to_csv('G:/Informatica/IA/enzimas/datasets/First Level/1l_background.csv')

In [ ]:
selected = pd.read_csv('G:/Informatica/IA/enzimas/datasets/First Level/1l_background.csv')
indexes = np.load('G:/Informatica/IA/enzimas/datasets/First Level/index_for_background.npy')
df = pd.read_csv(df_train_path)
df = df.iloc[indexes]
selected['sequence'] = df['sequence'].reset_index(drop=True)
selected.to_csv('G:/Informatica/IA/enzimas/datasets/First Level/1l_background.csv')